In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')
# Install necessary packages
!pip install mne plotly
import os
import numpy as np
!pip install colorama

import mne
import matplotlib.pyplot as plt
from colorama import Fore, Style, init

# Import libraries
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mne
import plotly.graph_objects as go
!pip install --upgrade mne

In [ ]:
import os
import numpy as np
import mne

# Define regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

group_20_25 = {'3', '15', '17', '19'}
group_30_35 = {'6', '7', '8', '10', '11', '12', '13', '14', '18', '20', '21', '22', '24', '25', '26', '27'}

subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    if subj not in group_20_25 and subj not in group_30_35:
        continue

    window_start_min = 20 if subj in group_20_25 else 30
    window_start = window_start_min * 60
    window_end = window_start + 5 * 60

    print("\n" + "=" * 50)
    print(f"🧠 Starting Participant {subj} | Plotting {window_start_min}-{window_start_min + 5} min window")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        raw = mne.io.read_raw_fif(session_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Collect good segments
        segments = []
        t0 = 0
        for annot in raw.annotations:
            if 'BREAK' in annot['description'].upper():
                if t0 < annot['onset']:
                    segments.append(raw.copy().crop(tmin=t0, tmax=annot['onset']))
                t0 = annot['onset'] + annot['duration']
        if t0 < raw.times[-1]:
            segments.append(raw.copy().crop(tmin=t0, tmax=None))

        if not segments:
            print(f"⏩ Skipping: no good data after removing breaks.")
            continue

        good_raw = mne.concatenate_raws(segments)

        if window_end > good_raw.times[-1]:
            print(f"⏩ Skipping: no good data in {window_start_min}-{window_start_min + 5} min window after removing breaks.")
            continue

        for region_name, region_channels in regions.items():
            picks = [ch for ch in region_channels if ch in good_raw.ch_names]
            if not picks:
                print(f"⚠ No valid channels for {region_name}. Skipping.")
                continue

            dummy_index = 1
            while len(picks) < 20:
                dummy_name = f'dummy{dummy_index}'
                if dummy_name not in good_raw.ch_names:
                    dummy_data = np.zeros((1, good_raw.n_times))
                    dummy_info = mne.create_info([dummy_name], good_raw.info['sfreq'], ch_types='eeg')
                    dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                    good_raw.add_channels([dummy_raw], force_update_info=True)
                picks.append(dummy_name)
                dummy_index += 1

            good_raw.plot(
                picks=picks,
                n_channels=20,
                scalings='auto',
                start=window_start,
                duration=5 * 60,
                title=f"Participant {subj} | {session_file} | {region_name} ({window_start_min}-{window_start_min + 5} min)",
                group_by='original',
                show_scrollbars=True,
                show_scalebars=True
            )



In [ ]:
import os
import numpy as np
import mne

# Define regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

group_20_25 = {'3', '15', '17', '19'}
group_30_35 = {'6', '7', '8', '10', '11', '12', '13', '14', '18', '20', '21', '22', '24', '25', '26', '27'}

subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    if subj not in group_20_25 and subj not in group_30_35:
        continue

    window_start_min = 20 if subj in group_20_25 else 30
    print("\n" + "=" * 50)
    print(f"🧠 Starting Participant {subj} | Plotting {window_start_min}-{window_start_min + 5} min window")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        raw = mne.io.read_raw_fif(session_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Build good segments
        segments = []
        t0 = 0
        for annot in raw.annotations:
            if 'BREAK' in annot['description'].upper():
                if t0 < annot['onset']:
                    segments.append(raw.copy().crop(tmin=t0, tmax=annot['onset']))
                t0 = annot['onset'] + annot['duration']
        if t0 < raw.times[-1]:
            segments.append(raw.copy().crop(tmin=t0, tmax=None))

        if not segments:
            print(f"⏩ Skipping: no good data after removing breaks.")
            continue

        good_raw = mne.concatenate_raws(segments)

        print(f"✅ Good data duration: {good_raw.times[-1]/60:.2f} min")

        if good_raw.times[-1] < 5 * 60:
            print(f"⏩ Skipping: not enough good data for 5 min plot.")
            continue

        for region_name, region_channels in regions.items():
            picks = [ch for ch in region_channels if ch in good_raw.ch_names]
            if not picks:
                print(f"⚠ No valid channels for {region_name}. Skipping.")
                continue

            # Add dummy channels if < 20
            dummy_index = 1
            while len(picks) < 20:
                dummy_name = f'dummy{dummy_index}'
                if dummy_name not in good_raw.ch_names:
                    dummy_data = np.zeros((1, good_raw.n_times))
                    dummy_info = mne.create_info([dummy_name], good_raw.info['sfreq'], ch_types='eeg')
                    dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                    good_raw.add_channels([dummy_raw], force_update_info=True)
                picks.append(dummy_name)
                dummy_index += 1

            # Plot from start of cleaned data
            good_raw.plot(
                picks=picks,
                n_channels=20,
                scalings='auto',
                start=0,  # Start at beginning of good data
                duration=5 * 60,
                title=f"Participant {subj} | {session_file} | {region_name} (cleaned start + 5 min)",
                group_by='original',
                show_scrollbars=True,
                show_scalebars=True
            )


In [ ]:
import os
import numpy as np
import pandas as pd
import mne

# Load break summary
break_summary_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/break_summary.csv'
break_df = pd.read_csv(break_summary_path)

# Define regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Starting Participant {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])

    subj_breaks = break_df[break_df['participant'] == int(subj)]

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        raw = mne.io.read_raw_fif(session_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')
        raw.info["bads"] = []

        break_list = [(row.onset_sec, row.onset_sec + row.duration_sec) for _, row in subj_breaks.iterrows()]
        file_end = raw.times[-1]

        t0 = 0
        found_window = False
        while t0 + 300 <= file_end:
            window_ok = True
            for onset, offset in break_list:
                if onset < t0 + 300 and offset > t0:
                    window_ok = False
                    break
            if window_ok:
                found_window = True
                break
            t0 += 10

        if not found_window:
            print(f"⏩ No 5-min clean window found for {subj} in {session_file}")
            continue

        print(f"✅ Plotting clean window: {t0/60:.2f}–{(t0+300)/60:.2f} min")

        for region_name, region_channels in regions.items():
            picks = [ch for ch in region_channels if ch in raw.ch_names]
            if not picks:
                print(f"No valid channels for {region_name}. Skipping.")
                continue

            dummy_index = 1
            while len(picks) < 20:
                dummy_name = f'dummy{dummy_index}'
                if dummy_name not in raw.ch_names:
                    dummy_data = np.zeros((1, raw.n_times))
                    dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
                    dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                    raw.add_channels([dummy_raw], force_update_info=True)  # 🟢 The key fix
                picks.append(dummy_name)
                dummy_index += 1

            raw.plot(
                picks=picks,
                n_channels=20,
                scalings=3e-4,
                start=t0,
                duration=30,
                title=f"{session_file} - {region_name} (clean {t0/60:.2f}–{(t0+300)/60:.2f} min)",
                group_by='original',
                show_scrollbars=False,
                show_scalebars=False
            )


In [ ]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt

# Set root path
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Set correlation threshold
correlation_threshold = 0.3

# Get and sort participant directories
subject_dirs = sorted(
    [d for d in os.listdir(root_dir) if d.isdigit()],
    key=lambda x: int(x)
)

# Process all participants
for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Processing Subject {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)

        # Load EEG data
        raw = mne.io.read_raw_fif(session_path, preload=True)
        print(f"Processing: {session_path}")

        # Set montage
        raw.set_montage('GSN-HydroCel-128')
        raw.info["bads"] = []

        # Pick EEG channels
        picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
        data, _ = raw[picks]
        ch_names = [raw.ch_names[i] for i in picks]

        # Compute correlation matrix
        corr_matrix = np.corrcoef(data)

        # Calculate mean correlation for each channel
        mean_corrs = []
        for i in range(corr_matrix.shape[0]):
            mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
            mean_corrs.append(mean_corr)

        # Identify bad channels
        bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
        bad_channels = [ch_names[i] for i in bad_channel_indices]

        # Output results
        print(f"\nSession: {session_file}")
        print(f"Total EEG channels: {len(ch_names)}")
        print(f"Correlation threshold: {correlation_threshold}")
        print(f"Detected bad channels (mean corr < {correlation_threshold}):")
        print(bad_channels)

        # Plot correlation matrix
        plt.figure(figsize=(16, 14))
        plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
        plt.colorbar(label='Correlation Coefficient')
        plt.title(f"Correlation Matrix - Subject {subj} {session_file}\n"
                  f"Potential bad channels: {', '.join(bad_channels) if bad_channels else 'None'}")
        plt.xticks(
            ticks=np.arange(len(ch_names)),
            labels=ch_names,
            rotation=90,
            fontsize=6,
            ha='center'
        )
        plt.yticks(
            ticks=np.arange(len(ch_names)),
            labels=ch_names,
            fontsize=6
        )
        plt.xlabel("Channel")
        plt.ylabel("Channel")
        plt.tight_layout()
        plt.show()


In [ ]:
import os
import mne

# Set root path
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Get participant directories
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Running ICA for Subject {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        raw = mne.io.read_raw_fif(session_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')
        raw.info['bads'] = []

        # Run ICA
        ica = mne.preprocessing.ICA(random_state=97, max_iter='auto')
        ica.fit(raw)

        # Plot all components topomap
        print(f"✅ Plotting ICA topomaps for {session_file}")
        ica.plot_components(inst=raw, show=True)

        # Plot detailed properties for each component
        print(f"✅ Plotting ICA properties (time series, spectrum) for {session_file}")
        for comp_idx in range(ica.n_components_):
            ica.plot_properties(raw, picks=comp_idx)


In [ ]:
import os
import mne

def run_ica_and_plot(participant_id, root_dir):
    """
    Run ICA on the given participant's _filtered_raw.fif files,
    plot components topomaps and time series/spectrum properties.
    """
    subj_path = os.path.join(root_dir, str(participant_id))
    if not os.path.isdir(subj_path):
        print(f"❌ Participant {participant_id} directory not found.")
        return

    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])
    if not session_files:
        print(f"❌ No _filtered_raw.fif files found for participant {participant_id}.")
        return

    print("\n" + "=" * 50)
    print(f"🧠 Running ICA for Subject {participant_id}")
    print("=" * 50 + "\n")

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        raw = mne.io.read_raw_fif(session_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')
        raw.info['bads'] = []

        # Run ICA
        ica = mne.preprocessing.ICA(random_state=97, max_iter='auto')
        ica.fit(raw)

        # Plot all components topomap
        print(f"✅ Plotting ICA topomaps for {session_file}")
        ica.plot_components(inst=raw, show=True)

        # Plot detailed properties for each component
        print(f"✅ Plotting ICA properties (time series, spectrum) for {session_file}")
        for comp_idx in range(ica.n_components_):
            ica.plot_properties(raw, picks=comp_idx)

# Example usage
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Call for individual participant
run_ica_and_plot(3, root_dir)


In [ ]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt

# Set root path
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Set correlation threshold
correlation_threshold = 0.3

# Get and sort participant directories
subject_dirs = sorted(
    [d for d in os.listdir(root_dir) if d.isdigit()],
    key=lambda x: int(x)
)

# Process all participants
for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Processing Subject {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_filtered_raw.fif')])

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)

        # Load EEG data
        raw = mne.io.read_raw_fif(session_path, preload=True)
        print(f"Processing: {session_path}")

        # Set montage
        raw.set_montage('GSN-HydroCel-128')
        raw.info["bads"] = []

        # Pick EEG channels
        picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
        data, _ = raw[picks]
        ch_names = [raw.ch_names[i] for i in picks]

        # Compute correlation matrix
        corr_matrix = np.corrcoef(data)

        # Calculate mean correlation for each channel
        mean_corrs = []
        for i in range(corr_matrix.shape[0]):
            mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
            mean_corrs.append(mean_corr)

        # Identify bad channels
        bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
        bad_channels = [ch_names[i] for i in bad_channel_indices]

        # Output results
        print(f"\nSession: {session_file}")
        print(f"Total EEG channels: {len(ch_names)}")
        print(f"Correlation threshold: {correlation_threshold}")
        print(f"Detected bad channels (mean corr < {correlation_threshold}):")
        print(bad_channels)

        # Plot correlation matrix
        plt.figure(figsize=(16, 14))
        plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
        plt.colorbar(label='Correlation Coefficient')
        plt.title(f"Correlation Matrix - Subject {subj} {session_file}\n"
                  f"Potential bad channels: {', '.join(bad_channels) if bad_channels else 'None'}")
        plt.xticks(
            ticks=np.arange(len(ch_names)),
            labels=ch_names,
            rotation=90,
            fontsize=6,
            ha='center'
        )
        plt.yticks(
            ticks=np.arange(len(ch_names)),
            labels=ch_names,
            fontsize=6
        )
        plt.xlabel("Channel")
        plt.ylabel("Channel")
        plt.tight_layout()
        plt.show()